### PSMNIST Classification with S4


In [ ]:
from __future__ import annotations
from typing import Dict, Any

import torch
import os
from pathlib import Path

from psmnist_task import PSMNISTTask
from src.train_utils.trainer import Trainer

## Configuration Setup

Using unified block factory from src.utils.block_factory.
This provides consistent block configuration across all experiments.


In [ ]:
from src.utils.block_factory import make_s4_block_cfg_ctor

## Hyperparameters & Data Paths

In [ ]:
current_dir = Path.cwd()
project_root = current_dir.parent.parent.parent
data_root = str(project_root / "src" / "datasets" / "psmnist" / "data")


def get_args() -> Dict[str, Any]:
    args: Dict[str, Any] = {
        "data_root": data_root,
        "batch": 128,
        "data_loader_kwargs": {
            "num_workers": 0,
            "use_permutation": True,
            "permutation_seed": 42,
            "normalize": "standard",
            "pin_memory": False,
            "persistent_workers": False,
        },

        "epochs": 50,
        "lr": 1e-3,
        "wd": 1e-4,
        "amp": False,
        "save_dir": "./runs/psmnist_s4_task",
        "warmup_epochs": 5,
        "patience": 5,
        "min_delta": 0.001,
        "early_key": "accuracy",

        "d_model": 128,
        "depth": 2,
        "dropout": 0.1,
        "mlp_ratio": 2.0,
        "droppath_final": 0.0,
        "layerscale_init": 0.0,
        "residual_gain": 1.0,
        "pool": "mean",
    }

    args["block_cfg_ctor"] = make_s4_block_cfg_ctor(
        dropout=args["dropout"],
        mlp_ratio=args["mlp_ratio"],
        droppath_final=args["droppath_final"],
        layerscale_init=args["layerscale_init"],
        residual_gain=args["residual_gain"],
        pool=args["pool"],
    )

    if torch.backends.mps.is_available():
        args["device"] = torch.device("mps")
        print("Using MPS (Apple Silicon)")
    elif torch.cuda.is_available():
        args["device"] = torch.device("cuda")
        print("Using CUDA")
    else:
        args["device"] = torch.device("cpu")
        args["amp"] = False
        print("Using CPU (slower)")

    return args


args = get_args()

print(f"Data root: {data_root}")
print(f"Save directory: {args['save_dir']}")

## Training

In [ ]:
task = PSMNISTTask()

if torch.backends.mps.is_available():
    torch.mps.set_per_process_memory_fraction(0.9)

print("\n" + "="*60)
print("INITIALIZING S4 MODEL FOR PSMNIST")
print("="*60)

trainer = Trainer(args=args, task=task)

print("\nStarting training...")
best_metric, ckpt_path = trainer.fit()

print("\n" + "="*60)
print("TRAINING COMPLETE")
print("="*60)
print(f"Best {trainer.early_key}: {best_metric:.4f}")
print(f"Checkpoint saved: {ckpt_path}")
print("="*60 + "\n")

In [ ]:
from src.utils.visualization import plot_classification_history as plot_history
from src.utils.checkpoint import load_trainer_from_checkpoint

trainer = load_trainer_from_checkpoint(
    checkpoint_path=args["save_dir"] + "/best.pt",
    args=args,
    task=PSMNISTTask(),
)

history = trainer.history

plot_history(history, model_name="S4")

In [ ]:
from src.utils.common import print_model_details

print_model_details(model=trainer.model)

## Evaluation

Load the best checkpoint and evaluate on the test set.
This provides the final accuracy metric for the S4 model on PSMNIST.


In [ ]:
from src.eval.eval_utils import evaluate_classification_model as evaluate_best_model

print("Evaluating best model on test set...")
logits_test, labels_test = evaluate_best_model(
    args=args,
    task=PSMNISTTask(),
    best_model_path=f"{args['save_dir']}/best.pt",
    num_classes=10,
    use_test_set=True,
)

### Evaluation on different model sizes - equal to LMU

In [ ]:
from src.eval.eval_utils import evaluate_classification_model as evaluate_best_model
from src.utils.common import print_model_details
from src.utils.checkpoint import load_trainer_from_checkpoint

equal_params_args = args.copy()
equal_params_args["d_model"] = 176
equal_params_args["depth"] = 4
equal_params_args['save_dir'] = "./runs/psmnist_s4_task_equal_params"

trainer = load_trainer_from_checkpoint(
    checkpoint_path=equal_params_args["save_dir"] + "/best.pt",
    args=equal_params_args,
    task=PSMNISTTask(),
)

plot_history(trainer.history, model_name="S4")

logits_test, labels_test = evaluate_best_model(
    args=equal_params_args,
    task=PSMNISTTask(),
    best_model_path=f"{equal_params_args['save_dir']}/best.pt",
    num_classes=10,
    use_test_set=True,
)

print_model_details(model=trainer.model)

### Changes in dataset length

In [ ]:
from src.eval.eval_utils import evaluate_classification_model as evaluate_best_model

if torch.backends.mps.is_available():
    torch.mps.set_per_process_memory_fraction(0.9)

for frac in [0.1, 0.25, 0.5]:
    print(f"\nTraining with fraction: {frac}")

    frac_args = get_args()
    frac_args["fraction"] = frac
    frac_args["save_dir"] = f"./runs/psmnist_s4_task_frac_{int(frac*100)}"

    trainer = Trainer(args=frac_args, task=PSMNISTTask())
    best_metric, best_path = trainer.fit()

    print(f"\nTraining complete for fraction {frac}! Best validation {trainer.early_key}: {best_metric:.4f}")
    print(f"Best model saved to: {best_path}")

    history = trainer.history

    plot_history(history, model_name="S4")

    logits_test, labels_test = evaluate_best_model(
        args=frac_args,
        task=PSMNISTTask(),
        best_model_path=best_path,
        num_classes=10,
        use_test_set=True,
    )

# SMNIST with S4
We can also run the same setup for the SMNIST task by changing the permutation to false.

In [ ]:
smnist_args = get_args()

smnist_args["data_loader_kwargs"]["use_permutation"] = False
smnist_args["save_dir"] = "../smnist/runs/smnist_s4_task"

print("=" * 70)
print("Training on SMNIST (Sequential MNIST - no permutation)")
print("=" * 70)

### Training SMNIST

In [ ]:
smnist_task = PSMNISTTask()

if torch.backends.mps.is_available():
    torch.mps.set_per_process_memory_fraction(0.9)

print("\n" + "="*60)
print("INITIALIZING S4 MODEL FOR SMNIST")
print("="*60)

smnist_trainer = Trainer(args=smnist_args, task=smnist_task)

print("\nStarting training...")
best_metric_smnist, ckpt_path_smnist = smnist_trainer.fit()

print("\n" + "="*60)
print("TRAINING COMPLETE")
print("="*60)
print(f"Best {smnist_trainer.early_key}: {best_metric_smnist:.4f}")
print(f"Checkpoint saved: {ckpt_path_smnist}")
print("="*60 + "\n")

In [ ]:
from src.utils.checkpoint import load_trainer_from_checkpoint

smnist_trainer = load_trainer_from_checkpoint(
    checkpoint_path=smnist_args["save_dir"] + "/best.pt",
    args=smnist_args,
    task=PSMNISTTask(),
)

smnist_history = smnist_trainer.history

plot_history(smnist_history, model_name="S4 (SMNIST)")

In [ ]:
from src.utils.common import print_model_details

print_model_details(model=smnist_trainer.model)

### Evaluation on SMNIST

In [ ]:
print("\n" + "=" * 70)
print("Evaluating SMNIST on Test Set")
print("=" * 70)

logits_test_smnist, labels_test_smnist = evaluate_best_model(
    args=smnist_args,
    task=PSMNISTTask(),
    best_model_path=f"{smnist_args['save_dir']}/best.pt",
    num_classes=10,
    use_test_set=True,
)

### Changes in dataset length (SMNIST)
Test how the S4 model performs with different fractions of the SMNIST training data.

In [ ]:
if torch.backends.mps.is_available():
    torch.mps.set_per_process_memory_fraction(0.9)

for frac in [0.1, 0.25, 0.5]:
    print(f"\nTraining SMNIST with fraction: {frac}")

    smnist_frac_args = get_args()
    smnist_frac_args["fraction"] = frac
    smnist_frac_args["save_dir"] = f"../smnist/runs/smnist_s4_task_frac_{int(frac*100)}"

    smnist_frac_trainer = Trainer(args=smnist_frac_args, task=PSMNISTTask())
    best_metric_frac, best_path_frac = smnist_frac_trainer.fit()

    print(f"\nTraining complete for SMNIST fraction {frac}! Best validation {smnist_frac_trainer.early_key}: {best_metric_frac:.4f}")
    print(f"Best model saved to: {best_path_frac}")

    smnist_frac_history = smnist_frac_trainer.history

    plot_history(smnist_frac_history, model_name=f"S4 (SMNIST) - {int(frac*100)}% data")

    logits_test_frac, labels_test_frac = evaluate_best_model(
        args=smnist_frac_args,
        task=PSMNISTTask(),
        best_model_path=best_path_frac,
        num_classes=10,
        use_test_set=True,
    )